# Science LLM: Google Colab Pipeline (T4 GPU)

### Google Drive Structure:
- **PDFs and Text Files**: `drive/MyDrive/Science LLM/Dataset`
- **Checkpoints**: `drive/MyDrive/Science LLM/checkpoints`
- **Trained Model**: `drive/MyDrive/Science LLM/models/SCLLM-v<version>`
- **Config & State**: `drive/MyDrive/Science LLM/config/data.json` (tracks restart/resume, training runs, versions, and test logs)

In [ ]:
# 1. Verify GPU Allocation
!nvidia-smi

In [ ]:
# 2. Install Fast Training Dependencies (Unsloth + HuggingFace Ecosystem)
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets transformers pymupdf pypdf evaluate

In [ ]:
# 3. Mount Google Drive and Initialize Directory Hierarchy
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

BASE_DIR = Path("/content/drive/MyDrive/Science LLM")
DATASET_DIR = BASE_DIR / "Dataset"
CHECKPOINTS_DIR = BASE_DIR / "checkpoints"
MODELS_DIR = BASE_DIR / "models"
CONFIG_DIR = BASE_DIR / "config"
DATA_JSON_PATH = CONFIG_DIR / "data.json"

for p in [DATASET_DIR, CHECKPOINTS_DIR, MODELS_DIR, CONFIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Directory Structure Initialized on Google Drive:")
print(f"- Dataset:     {DATASET_DIR}")
print(f"- Checkpoints: {CHECKPOINTS_DIR}")
print(f"- Models:      {MODELS_DIR}")
print(f"- Config:      {DATA_JSON_PATH}")

In [ ]:
# 4. Initialize or Load data.json State Tracking
import json
import time

VERSION = "1.0.0"

if DATA_JSON_PATH.exists():
    with open(DATA_JSON_PATH, "r", encoding="utf-8") as f:
        state = json.load(f)
    print(f"Loaded state from {DATA_JSON_PATH} (Active Version: {state.get('current_version')})")
else:
    state = {
        "project_name": "Science LLM",
        "current_version": VERSION,
        "model_name_tag": f"SCLLM-v{VERSION}",
        "base_model": "Qwen/Qwen2.5-3B",
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "paths": {
            "dataset": str(DATASET_DIR),
            "checkpoints": str(CHECKPOINTS_DIR),
            "models": str(MODELS_DIR),
            "config": str(CONFIG_DIR)
        },
        "checkpoints": {
            "latest_stage": None,
            "latest_step": 0,
            "latest_checkpoint_path": None,
            "can_resume": False,
            "history": []
        },
        "training_runs": [],
        "evaluation_benchmarks": {"test_results": []}
    }
    with open(DATA_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2)
    print(f"Created new state tracker at {DATA_JSON_PATH}")

## 5. Data Ingestion: Science Books & Articles
Place any PDF textbooks or `.txt` articles inside `/content/drive/MyDrive/Science LLM/Dataset`.
The cell below parses multi-column PDFs, cleans headers/references, preserves LaTeX math, and packs tokens.

In [ ]:
import re
from datasets import Dataset, load_dataset

# Check for uploaded files in Dataset directory
pdf_files = list(DATASET_DIR.glob("**/*.pdf"))
txt_files = list(DATASET_DIR.glob("**/*.txt"))
print(f"Found {len(pdf_files)} PDFs and {len(txt_files)} text files in {DATASET_DIR}")

corpus_records = []
if pdf_files:
    import fitz
    for p in pdf_files:
        doc = fitz.open(str(p))
        blocks = []
        for page in doc:
            for b in page.get_text("blocks"):
                if b[6] == 0 and b[4].strip():
                    blocks.append(b[4].strip())
        full_text = "\n\n".join(blocks)
        corpus_records.append({"text": full_text, "source": p.name})
        print(f"Extracted {p.name} ({len(full_text)} chars)")

# If no local PDFs, use SciQ as default
if not corpus_records:
    print("No local PDFs found; streaming SciQ science dataset...")
    sciq = load_dataset("sciq", split="train[:6000]")
    train_dataset = sciq.map(lambda x: {"text": f"Topic: {x['question']}\nExplanation: {x['support']}\nFact: {x['correct_answer']}"})
else:
    train_dataset = Dataset.from_list(corpus_records)

print(f"Dataset prepared. Total samples: {len(train_dataset)}")

## 6. Stage 1: Continued Pre-Training (DAPT) with Automatic Resume
Automatically inspects `drive/Science LLM/config/data.json` and checkpoints directory for existing checkpoints to continue seamlessly.

In [ ]:
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments, TrainerCallback

MODEL_NAME = state.get("base_model", "Qwen/Qwen2.5-3B")
DAPT_OUTPUT = str(CHECKPOINTS_DIR / "stage1_dapt")

# Check for existing checkpoint to resume
resume_ck = None
if state.get("checkpoints", {}).get("latest_checkpoint_path") and os.path.exists(state["checkpoints"]["latest_checkpoint_path"]):
    resume_ck = state["checkpoints"]["latest_checkpoint_path"]
    print(f"Resuming training from checkpoint: {resume_ck}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=torch.float16,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

class ColabSyncCallback(TrainerCallback):
    def on_save(self, args, trainer_state, control, **kwargs):
        ck_path = os.path.join(args.output_dir, f"checkpoint-{trainer_state.global_step}")
        state["checkpoints"]["latest_stage"] = "stage1_dapt"
        state["checkpoints"]["latest_step"] = trainer_state.global_step
        state["checkpoints"]["latest_checkpoint_path"] = ck_path
        state["checkpoints"]["can_resume"] = True
        with open(DATA_JSON_PATH, "w", encoding="utf-8") as f:
            json.dump(state, f, indent=2)
        print(f"\n[Synced Checkpoint {trainer_state.global_step} to data.json]")

training_args = TrainingArguments(
    output_dir=DAPT_OUTPUT,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_ratio=0.05,
    max_steps=300,
    learning_rate=1e-4,
    fp16=True,
    bf16=False,
    logging_steps=20,
    optim="paged_adamw_8bit",
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=True,
    args=training_args,
    callbacks=[ColabSyncCallback()]
)

trainer.train(resume_from_checkpoint=resume_ck)

final_dapt_adapter = os.path.join(DAPT_OUTPUT, "final_dapt_adapter")
model.save_pretrained(final_dapt_adapter)
tokenizer.save_pretrained(final_dapt_adapter)
print(f"Stage 1 adapter saved to: {final_dapt_adapter}")

## 7. Stage 2: Supervised Fine-Tuning (SFT) & Scientific Reasoning Alignment

In [ ]:
from trl import DataCollatorForCompletionOnlyLM

SYSTEM_PROMPT = """You are an expert scientific researcher. Provide rigorous, step-by-step evidence-based reasoning using precise scientific nomenclature and LaTeX equations."""

sciq = load_dataset("sciq", split="train")

def format_sft(ex):
    user_q = ex['question']
    if ex.get('support'):
        user_q += f"\nContext:\n{ex['support']}"
    return {
        "text": f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{user_q}<|im_end|>\n<|im_start|>assistant\n{ex['correct_answer']}<|im_end|>"
    }

sft_dataset = sciq.map(format_sft)
response_template = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(response_template=response_template, tokenizer=tokenizer)

SFT_OUTPUT = str(CHECKPOINTS_DIR / "stage2_sft")

sft_args = TrainingArguments(
    output_dir=SFT_OUTPUT,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_ratio=0.05,
    max_steps=200,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    logging_steps=20,
    optim="paged_adamw_8bit",
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none"
)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_dataset,
    dataset_text_field="text",
    data_collator=collator,
    max_seq_length=2048,
    args=sft_args
)

sft_trainer.train()

final_sft_adapter = os.path.join(SFT_OUTPUT, "final_sft_adapter")
model.save_pretrained(final_sft_adapter)
tokenizer.save_pretrained(final_sft_adapter)
print(f"Stage 2 SFT adapter saved to: {final_sft_adapter}")

## 8. Export Versioned Standalone Model: `SCLLM-v<version>`
Merges the trained LoRA adapter directly into FP16 and exports to `drive/MyDrive/Science LLM/models/SCLLM-v<version>`.
Logs complete model info into `data.json`.

In [ ]:
VERSION_TAG = f"SCLLM-v{state.get('current_version', '1.0.0')}"
MODEL_SAVE_PATH = MODELS_DIR / VERSION_TAG
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

print(f"Merging and saving standalone model to: {MODEL_SAVE_PATH}...")
model.save_pretrained_merged(str(MODEL_SAVE_PATH), tokenizer, save_method="merged_16bit")

# Update data.json
state["trained_models"] = state.get("trained_models", [])
state["trained_models"].append({
    "tag": VERSION_TAG,
    "version": state.get("current_version", "1.0.0"),
    "path": str(MODEL_SAVE_PATH),
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
})
with open(DATA_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(state, f, indent=2)

print(f"Successfully exported {VERSION_TAG} to Google Drive!")

## 9. Benchmark Evaluation & Interactive Science Testing

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = "Explain the biochemical mechanism of CRISPR-Cas9 in targeted DNA cleavage, specifically how PAM sequence and sgRNA guide Cas9."
formatted = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{test_prompt}<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer([formatted], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.3, top_p=0.9)
response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("="*70)
print(f"Prompt: {test_prompt}")
print("-"*70)
print(response_text)
print("="*70)

# Log test result into data.json
state["evaluation_benchmarks"]["test_results"].append({
    "prompt": test_prompt,
    "response": response_text,
    "model_tag": VERSION_TAG,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
})
with open(DATA_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(state, f, indent=2)
print("Logged test response to data.json.")